In [1]:
# ============================================================
# Google Colab用 - 出退判定◯：◯以外を所属名別に分割
# 使い方:
# 1. このセルを実行 → ファイルをアップロード
# 2. 日付を選択（任意・未選択なら全期間）して「実行」ボタンを押す
# 3. 所属名ごとのExcelファイルZIP＋画像のみZIPをダウンロード
# ============================================================

from google.colab import files
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Border, Side, PatternFill, Font, Alignment, Color
from openpyxl.utils import get_column_letter
import ipywidgets as widgets
from IPython.display import display
import os, re, zipfile, copy, datetime, io
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
from PIL import Image, ImageChops

# ── アップロード ──────────────────────────────────────────
uploaded = files.upload()
src = list(uploaded.keys())[0]
print(f"アップロード完了: {src}")

# ── 日本語フォント設定（Colab環境） ──────────────────────
!apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1
fm.fontManager.__init__()
_jp_fonts = [f.name for f in fm.fontManager.ttflist if 'Noto' in f.name and 'CJK' in f.name]
_font_name = _jp_fonts[0] if _jp_fonts else 'DejaVu Sans'
plt.rcParams['font.family'] = _font_name

# ── ファイルから日付一覧を取得 ────────────────────────────
_wb_tmp = load_workbook(src, read_only=True)
_ws_tmp = _wb_tmp.active

# ヘッダー行からキー列のインデックスを自動検出
_header_row = next(_ws_tmp.iter_rows(min_row=1, max_row=1, values_only=True), None)
_headers = [str(v).strip() if v is not None else '' for v in _header_row]

def _find_col(candidates):
    """候補名リストのいずれかに部分一致する列インデックスを返す"""
    for i, h in enumerate(_headers):
        for c in candidates:
            if c in h:
                return i
    return None

_col_date  = _find_col(['日時', '日付', 'date', 'Date'])
_col_store = _find_col(['所属名', '店舗名', '部署名'])
_col_judge = _find_col(['出退判定', '判定'])

print(f"検出列 → 日付:{_col_date}列目({_headers[_col_date] if _col_date is not None else 'なし'})  "
      f"所属名:{_col_store}列目({_headers[_col_store] if _col_store is not None else 'なし'})  "
      f"判定:{_col_judge}列目({_headers[_col_judge] if _col_judge is not None else 'なし'})")

if _col_date is None:
    print("⚠ 日付列が見つかりませんでした。ヘッダー名を確認してください。")
    print(f"  全ヘッダー: {_headers}")
    _dates = []
else:
    _dates = sorted(set(
        str(row[_col_date])[:10] for row in _ws_tmp.iter_rows(min_row=2, values_only=True)
        if row[_col_date] and re.match(r'\d{4}-\d{2}-\d{2}', str(row[_col_date]))
    ))
_wb_tmp.close()

if not _dates:
    print("⚠ 日付データが見つかりませんでした。")
else:
    print(f"ファイル内の日付: {_dates[0]} ～ {_dates[-1]}  ({len(_dates)}日分)")

# ── 日付選択UI ────────────────────────────────────────────
_ALL = '（指定なし・全期間）'
_opts = [_ALL] + _dates

_mode = widgets.ToggleButtons(
    options=[('単日選択', 'single'), ('範囲指定', 'range')],
    description='モード:',
    button_style='info',
)
_date_single = widgets.Dropdown(
    options=_opts, value=_ALL, description='日付:',
    layout=widgets.Layout(width='320px')
)
_date_from = widgets.Dropdown(
    options=_opts, value=_ALL, description='開始日:',
    layout=widgets.Layout(width='320px')
)
_date_to = widgets.Dropdown(
    options=_opts, value=_ALL, description='終了日:',
    layout=widgets.Layout(width='320px')
)
_single_box = widgets.VBox([_date_single])
_range_box  = widgets.VBox([_date_from, _date_to])
_content    = widgets.VBox([_single_box])
_btn = widgets.Button(
    description='実行',
    button_style='success',
    layout=widgets.Layout(width='160px', margin='10px 0 0 0')
)
display(widgets.VBox([_mode, _content, _btn]))

def _on_mode_change(change):
    _content.children = [_single_box] if change['new'] == 'single' else [_range_box]
_mode.observe(_on_mode_change, names='value')

# ── スタイル設定 ──────────────────────────────────────────
MATCH_OK        = '◯ ： ◯'
EXCLUDE_HEADERS = {'従業員コード', '所定時間', '残業時間', '所属コード'}

m      = re.search(r'(出退勤.*)', src)
suffix = m.group(1) if m else src

wb_src = load_workbook(src)
ws_src = wb_src.active

all_visible  = [c for c in range(1, ws_src.max_column + 1)
                if not (ws_src.column_dimensions.get(get_column_letter(c)) and
                        ws_src.column_dimensions[get_column_letter(c)].hidden)]
visible_cols = [c for c in all_visible
                if (ws_src.cell(1, c).value or '').replace('\n', '').strip() not in EXCLUDE_HEADERS]

thin        = Side(style='thin')
border      = Border(left=thin, right=thin, top=thin, bottom=thin)
HEADER_FILL = PatternFill(fill_type='solid', fgColor=Color(theme=7, tint=0.7999816888943144))

def copy_cell_style(src_cell, dst_cell, is_header=False):
    if is_header:
        dst_cell.fill = HEADER_FILL
    elif src_cell.fill and src_cell.fill.fill_type not in (None, 'none'):
        dst_cell.fill = copy.copy(src_cell.fill)
    src_bold = src_cell.font.bold if src_cell.font else False
    dst_cell.font = Font(name='游ゴシック', size=11, bold=src_bold if is_header else False)
    if src_cell.alignment:
        dst_cell.alignment = copy.copy(src_cell.alignment)
    dst_cell.border = border
    if isinstance(src_cell.value, datetime.time):
        dst_cell.number_format = 'h:mm'

# ── PNG生成ヘルパー ───────────────────────────────────────
MAX_ROWS_PER_IMAGE = 50

def save_table_as_png(df, out_path, title=''):
    """DataFrameをmatplotlibのテーブルとしてPNG保存。行数が多い場合は分割。"""
    n_cols = len(df.columns)
    col_labels = list(df.columns)
    all_rows = df.values.tolist()
    chunks = [all_rows[i:i+MAX_ROWS_PER_IMAGE] for i in range(0, max(len(all_rows), 1), MAX_ROWS_PER_IMAGE)]
    paths = []
    for idx, chunk in enumerate(chunks):
        n_rows = len(chunk)
        fig_h = max(1.2, 0.45 * n_rows + 0.8)
        fig_w = max(6, 1.5 * n_cols)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.axis('off')
        tbl = ax.table(
            cellText=chunk,
            colLabels=col_labels,
            loc='center',
            cellLoc='center',
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9)
        tbl.auto_set_column_width(col=list(range(n_cols)))
        for (r, c), cell in tbl.get_celld().items():
            if r == 0:
                cell.set_facecolor('#BDD7EE')
                cell.set_height(cell.get_height() * 2)
                cell.set_text_props(fontweight='bold', wrap=True)
            else:
                cell.set_facecolor('#FFFFFF' if r % 2 == 1 else '#F2F2F2')
            cell.set_edgecolor('#AAAAAA')

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=150, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf).convert('RGB')
        bg   = Image.new('RGB', img.size, (255, 255, 255))
        diff = ImageChops.difference(img, bg)
        bbox = diff.getbbox()
        if bbox:
            img = img.crop(bbox)

        p = out_path if len(chunks) == 1 else out_path.replace('.png', f'_{idx+1}.png')
        img.save(p, dpi=(150, 150))
        paths.append(p)
    return paths

def excel_to_df(xlsx_path):
    """openpyxlで読み込んでDataFrameに変換（time型を文字列化）"""
    wb = load_workbook(xlsx_path, read_only=True, data_only=True)
    ws = wb.active
    rows = list(ws.iter_rows(values_only=True))
    wb.close()
    if not rows:
        return pd.DataFrame()
    headers = [str(v) if v is not None else '' for v in rows[0]]
    data = []
    for row in rows[1:]:
        data.append([
            v.strftime('%H:%M') if isinstance(v, datetime.time)
            else (str(v) if v is not None else '')
            for v in row
        ])
    return pd.DataFrame(data, columns=headers)

# ── 実行ボタン処理 ────────────────────────────────────────
def _run(b):
    # 列インデックスの確認
    if _col_date is None or _col_store is None or _col_judge is None:
        print("⚠ 必要な列（日付・所属名・判定）が検出できていません。処理を中止します。")
        return
    if not _dates:
        print("⚠ 日付データが読み込めていません。ファイルを確認してください。")
        return

    # 日付フィルタ範囲を決定
    if _mode.value == 'single':
        sel = _date_single.value
        date_from = date_to = None if sel == _ALL else sel
    else:
        df = _date_from.value;  dt = _date_to.value
        date_from = None if df == _ALL else df
        date_to   = None if dt == _ALL else dt

    if date_from is None and date_to is None:
        label = '全期間'
    elif date_from == date_to:
        label = date_from
    else:
        # ★修正3: _dates が空でも安全に参照
        label = f"{date_from or _dates[0]}～{date_to or _dates[-1]}"
    print(f"\n対象期間: {label}")

    # フィルタ条件でグループ化（行番号保持）
    data_by_store = {}
    for src_row in ws_src.iter_rows(min_row=2):
        store = src_row[_col_store].value
        date  = src_row[_col_date].value
        judge = src_row[_col_judge].value
        if store is None or date is None:
            continue
        date_str = str(date)[:10]
        if date_from and date_str < date_from:
            continue
        if date_to and date_str > date_to:
            continue
        if str(judge) == MATCH_OK:
            continue
        store = str(store).strip()
        if store not in data_by_store:
            data_by_store[store] = []
        data_by_store[store].append(src_row[0].row)

    total = sum(len(v) for v in data_by_store.values())
    print(f"抽出行数: {total}行 / 所属名: {len(data_by_store)}店舗")
    if not data_by_store:
        print("⚠ 条件に一致するデータがありませんでした。")
        return

    # 出力フォルダ初期化
    out_dir = 'output_filtered'
    img_dir = 'output_images'
    for d in [out_dir, img_dir]:
        os.makedirs(d, exist_ok=True)
        for f in os.listdir(d):
            os.remove(os.path.join(d, f))

    all_png_paths = []

    # 所属名ごとに1ファイル作成
    for store, src_rows in data_by_store.items():
        wb_out = Workbook()
        ws_out = wb_out.active
        ws_out.title = ws_src.title
        wb_out.loaded_theme = wb_src.loaded_theme

        for out_c, orig_c in enumerate(visible_cols, start=1):
            sc = ws_src.cell(1, orig_c); dc = ws_out.cell(1, out_c)
            dc.value = sc.value
            copy_cell_style(sc, dc, is_header=(out_c != len(visible_cols)))

        for out_c, orig_c in enumerate(visible_cols, start=1):
            ol = get_column_letter(orig_c); dl = get_column_letter(out_c)
            if ol in ws_src.column_dimensions:
                ws_out.column_dimensions[dl].width = ws_src.column_dimensions[ol].width

        for out_r, src_r in enumerate(src_rows, start=2):
            for out_c, orig_c in enumerate(visible_cols, start=1):
                sc = ws_src.cell(src_r, orig_c); dc = ws_out.cell(out_r, out_c)
                dc.value = sc.value
                copy_cell_style(sc, dc)

        safe     = store.translate(str.maketrans('/\\*?:"<>|', '／＼＊？："＜＞｜'))
        out_name = f"{safe}_{suffix}"
        xlsx_path = os.path.join(out_dir, out_name)
        wb_out.save(xlsx_path)
        print(f"  保存: {out_name} ({len(src_rows)}行)", end='')

        # PNG生成
        try:
            df_img = excel_to_df(xlsx_path)
            df_img = df_img.iloc[:, :-1]   # 右端の凡例列を除外
            png_name = out_name.replace('.xlsx', '.png')
            png_path = os.path.join(img_dir, png_name)
            saved_pngs = save_table_as_png(df_img, png_path, title=safe)
            all_png_paths.extend(saved_pngs)
            print(f"  → 画像: {[os.path.basename(p) for p in saved_pngs]}")
        except Exception as e:
            print(f"  → 画像生成エラー: {e}")

    zip_label = label.replace('～', '_')

    # Excel ZIP
    zip_excel = src.replace('.xlsx', f'_{zip_label}_分割.zip')
    with zipfile.ZipFile(zip_excel, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fname in os.listdir(out_dir):
            zf.write(os.path.join(out_dir, fname), fname)
    print(f"\nExcel ZIP: {zip_excel}")
    files.download(zip_excel)

    # 画像専用 ZIP
    if all_png_paths:
        zip_images = src.replace('.xlsx', f'_{zip_label}_画像.zip')
        with zipfile.ZipFile(zip_images, 'w', zipfile.ZIP_DEFLATED) as zf:
            for p in all_png_paths:
                zf.write(p, os.path.basename(p))
        print(f"画像 ZIP: {zip_images}")
        files.download(zip_images)

    print("ダウンロード開始")

_btn.on_click(_run)


Saving KIDS店舗出退勤_2026_0625.xlsx to KIDS店舗出退勤_2026_0625.xlsx
アップロード完了: KIDS店舗出退勤_2026_0625.xlsx
検出列 → 日付:6列目(日時)  所属名:2列目(所属名)  判定:15列目(出退判定)
ファイル内の日付: 2026-06-01 ～ 2026-06-24  (24日分)



対象期間: 2026-06-23～2026-06-24
抽出行数: 215行 / 所属名: 98店舗
  保存: 001KM岩曽_出退勤_2026_0625.xlsx (1行)  → 画像: ['001KM岩曽_出退勤_2026_0625.png']
  保存: 004KM上戸祭_出退勤_2026_0625.xlsx (5行)  → 画像: ['004KM上戸祭_出退勤_2026_0625.png']
  保存: 008KM真岡_出退勤_2026_0625.xlsx (2行)  → 画像: ['008KM真岡_出退勤_2026_0625.png']
  保存: 009KM桜通り_出退勤_2026_0625.xlsx (2行)  → 画像: ['009KM桜通り_出退勤_2026_0625.png']
  保存: 012KM若松原_出退勤_2026_0625.xlsx (2行)  → 画像: ['012KM若松原_出退勤_2026_0625.png']
  保存: 013KM真岡東／013KP真岡東_出退勤_2026_0625.xlsx (3行)  → 画像: ['013KM真岡東／013KP真岡東_出退勤_2026_0625.png']
  保存: 014KM西原_出退勤_2026_0625.xlsx (1行)  → 画像: ['014KM西原_出退勤_2026_0625.png']
  保存: 015KM宮の内_出退勤_2026_0625.xlsx (1行)  → 画像: ['015KM宮の内_出退勤_2026_0625.png']
  保存: 016KM新小山駅東_出退勤_2026_0625.xlsx (2行)  → 画像: ['016KM新小山駅東_出退勤_2026_0625.png']
  保存: 018KM今泉新町／018KP今泉新町_出退勤_2026_0625.xlsx (4行)  → 画像: ['018KM今泉新町／018KP今泉新町_出退勤_2026_0625.png']
  保存: 021KM新宮の内／021KP新宮の内_出退勤_2026_0625.xlsx (1行)  → 画像: ['021KM新宮の内／021KP新宮の内_出退勤_2026_0625.png']
  保存: 024KM並木_出退勤_2026_0625.xlsx (3行)  → 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

画像 ZIP: KIDS店舗出退勤_2026_0625_2026-06-23_2026-06-24_画像.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ダウンロード開始
